### ANN Classification (Multi-class)

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("DateFruit_Dataset.csv")

In [ ]:
df.head() 

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [ ]:
df.shape # 34 features, 1 category

(898, 35)

In [6]:
X = df.drop("Class", axis=1)
y = df["Class"]

In [ ]:
df["Class"].unique() # 7 classes = 7 neurons in output layer

array(['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY'],
      dtype=object)

In [10]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

le = LabelEncoder()
y = le.fit_transform(y)

In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [13]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#### ANN

In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [21]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32) 

y_train_tensor = torch.tensor(y_train, dtype=torch.long) # cross entropy loss expects y_train & y_test => long type
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [23]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [25]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

#### Build our ANN model

In [ ]:
class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()
        
        self.model = nn.Sequential(
            nn.Linear(X.shape[1], 64),
            nn.ReLU(),
            
            nn.Linear(64, 64),
            nn.ReLU(),
            
            nn.Linear(64, 7), # do not need softmax when using CrossEntropyLoss
        )
        
    def forward(self, x):
        return self.model(x)

In [30]:
model = ANN()

# loss & optim
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

#### Training the ANN

In [32]:
epochs = 100

for epoch in range(epochs):
    model.train()
    
    running_loss = 0.0
    
    for xb, yb in train_loader:
        optimizer.zero_grad()
        
        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step() # params update
        
        running_loss += loss.item()
        
    train_loss = running_loss / len(train_loader)
    
    print(f"epoch = {epoch+1}/{epochs}, loss = {train_loss}")

epoch = 1/100, loss = 0.02194914749175634
epoch = 2/100, loss = 0.02147498317366547
epoch = 3/100, loss = 0.02151601847625621
epoch = 4/100, loss = 0.018668359773152548
epoch = 5/100, loss = 0.01872682329469725
epoch = 6/100, loss = 0.021554366746188505
epoch = 7/100, loss = 0.01819263730177899
epoch = 8/100, loss = 0.023649190203286707
epoch = 9/100, loss = 0.019204663153251877
epoch = 10/100, loss = 0.022630782332271338
epoch = 11/100, loss = 0.017474434308140822
epoch = 12/100, loss = 0.02503495277715442
epoch = 13/100, loss = 0.020273368142585714
epoch = 14/100, loss = 0.01603064582804623
epoch = 15/100, loss = 0.018183299897076642
epoch = 16/100, loss = 0.014406628051327298
epoch = 17/100, loss = 0.015479130113659345
epoch = 18/100, loss = 0.013401650266883813
epoch = 19/100, loss = 0.01501831847101288
epoch = 20/100, loss = 0.01904534392119588
epoch = 21/100, loss = 0.013307091182587496
epoch = 22/100, loss = 0.012752269551603367
epoch = 23/100, loss = 0.013709272040337648
epoch 

In [34]:
# Evaluate
model.eval()

total = 0
correct = 0

with torch.no_grad():
    for xb, yb in test_loader:
        outputs = model(xb) # 7 raw predicted values => tensor (using softmax)
        _, predicted = torch.max(outputs, 1)
        
        correct += (predicted == yb).sum().item()
        total += yb.size(0) # actual samples in each batch

print("total vals:", total)
print("correct vals:", correct)

print("accuracy:", correct/total * 100)
        

total vals: 180
correct vals: 169
accuracy: 93.88888888888889
